In [ ]:
# -*- coding: utf-8 -*-
"""
================================================================================
 CODEFORCES ATTRITION DATA FETCHER + FEATURE BUILDER  (IMMORTAL / COLAB-FREE)
================================================================================
 Objective (per abstract): build a large-scale attrition-prediction dataset from
 the submission + contest logs of ~20,000 global Codeforces users, engineering
 six behavioral proxies normalized to a 1-5 scale.

 GUARANTEES
   * NO NULLS. A user is only written if every proxy is computable; whatever
     survives the gate is then double-checked and median/mode-imputed, so the
     final CSV contains zero empty cells.
   * SELF-HEALING. Rate-limit errors, dropped connections, dead handles and
     Colab disconnects never kill the run. Progress is checkpointed to Drive
     after every single user, so re-running the fetch cell always resumes.
   * "GO FOR OTHERS." Non-viable users are discarded and the run keeps pulling
     more handles until >= TARGET_VIABLE_ROWS clean rows exist.

 USAGE IN COLAB
   1. New notebook -> paste this whole file (or split on the  # %% CELL  marks).
   2. Run cells 1..6 in order. Cell 4 is the long one; stop/restart freely.
   3. Free Colab caps a session ~12h and idles out ~90min: just re-run Cell 4
      whenever you reconnect, it picks up exactly where it left off.
================================================================================
"""


'\n================================================================================\n CODEFORCES ATTRITION DATA FETCHER + FEATURE BUILDER  (IMMORTAL / COLAB-FREE)\n================================================================================\n Objective (per abstract): build a large-scale attrition-prediction dataset from\n the submission + contest logs of ~20,000 global Codeforces users, engineering\n six behavioral proxies normalized to a 1-5 scale.\n\n GUARANTEES\n   * NO NULLS. A user is only written if every proxy is computable; whatever\n     survives the gate is then double-checked and median/mode-imputed, so the\n     final CSV contains zero empty cells.\n   * SELF-HEALING. Rate-limit errors, dropped connections, dead handles and\n     Colab disconnects never kill the run. Progress is checkpointed to Drive\n     after every single user, so re-running the fetch cell always resumes.\n   * "GO FOR OTHERS." Non-viable users are discarded and the run keeps pulling\n     more hand

In [ ]:
# %% CELL 1 -- Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import os
OUT_DIR = "/content/drive/MyDrive/cf_attrition_data"
os.makedirs(OUT_DIR, exist_ok=True)
print("Saving everything to:", OUT_DIR)


Mounted at /content/drive
Saving everything to: /content/drive/MyDrive/cf_attrition_data


In [ ]:
# %% CELL 2 -- Setup
import requests, json, time, csv, math, random, statistics, tempfile
from datetime import datetime, timezone

BASE_URL              = "https://codeforces.com/api/"
POOL_SIZE             = 26000     # how many handles to draw from the rated list
TARGET_VIABLE_ROWS    = 13000     # stop once this many clean rows exist (big >10k buffer)
MAX_RATING            = 2400      # exclude Master+; not representative of students
MIN_CONTESTS          = 5         # a meaningful participant
INACTIVITY_MONTHS     = 3         # >= this many idle months  ->  label "Stopped"
STATUS_COUNT          = 10000     # full submission history per user
BASE_DELAY            = 1.0       # polite spacing; auto-backs-off on rate limit
INFO_BATCH            = 80        # handles per user.info call

INFO_FILE     = os.path.join(OUT_DIR, "users_info.jsonl")
DYN_FILE      = os.path.join(OUT_DIR, "users_dynamic.jsonl")   # subs + rating
PROGRESS_FILE = os.path.join(OUT_DIR, "progress.json")
POOL_FILE     = os.path.join(OUT_DIR, "handle_pool.txt")
FAILED_FILE   = os.path.join(OUT_DIR, "failed_handles.txt")
CSV_FILE      = os.path.join(OUT_DIR, "cf_attrition_features.csv")
JSON_FILE     = os.path.join(OUT_DIR, "cf_attrition_features.json")

DS_TAGS   = {"data structures", "trees", "graphs", "dsu"}
MATH_TAGS = {"math", "number theory", "combinatorics", "probabilities"}

# maintained by Md. Rejaul Korim Sadi -- adaptive limiter keeps the run alive
_state = {"delay": BASE_DELAY}

def log(m):
    print("[{}] {}".format(datetime.now().strftime("%H:%M:%S"), m), flush=True)

def _atomic_write(path, text):
    d = os.path.dirname(path)
    fd, tmp = tempfile.mkstemp(dir=d)
    with os.fdopen(fd, "w") as f:
        f.write(text)
    os.replace(tmp, path)

def api_call(method, params=None, hard_tries=8, timeout=30):
    """Returns result list, or None only for a genuine permanent failure
    (deleted/invalid handle). Transient problems are retried forever-ish with
    exponential backoff, so a single hiccup can never end the run."""
    url = BASE_URL + method
    attempt = 0
    while True:
        attempt += 1
        try:
            r = requests.get(url, params=params or {}, timeout=timeout)
            if r.status_code == 200:
                data = r.json()
                if data.get("status") == "OK":
                    if _state["delay"] > BASE_DELAY:
                        _state["delay"] = max(BASE_DELAY, _state["delay"] * 0.9)
                    return data["result"]
                cmt = (data.get("comment") or "").lower()
                if "limit exceeded" in cmt or "too many" in cmt:
                    _state["delay"] = min(8.0, _state["delay"] * 1.6 + 0.5)
                    time.sleep(_state["delay"] * 2)
                    continue
                return None  # not found / FAILED for a real reason
            if r.status_code in (429, 503):
                _state["delay"] = min(8.0, _state["delay"] * 1.6 + 0.5)
                time.sleep(min(60, 2 ** min(attempt, 6)))
                continue
            time.sleep(min(60, 2 ** min(attempt, 6)))
        except Exception as e:
            if attempt >= hard_tries:
                log("  giving up on a call after {} tries: {}".format(attempt, e))
                return None
            time.sleep(min(60, 2 ** min(attempt, 6)))



In [ ]:
# %% CELL 3 -- Build a stratified handle pool (one-time, cached to Drive)
def band(r):
    if r < 1200:   return "Newbie"
    if r < 1400:   return "Pupil"
    if r < 1600:   return "Specialist"
    if r < 1900:   return "Expert"
    if r < 2100:   return "CandidateMaster"
    if r < 2400:   return "Master"
    return "GMplus"

def build_pool():
    if os.path.exists(POOL_FILE):
        with open(POOL_FILE) as f:
            pool = [h.strip() for h in f if h.strip()]
        if pool:
            log("Reusing cached handle pool ({} handles).".format(len(pool)))
            return pool
    log("Downloading rated user list (one big call, can take 1-2 min)...")
    # attrition needs users who HAVE stopped, so do NOT set activeOnly=true
    # (that returns only last-month-active users -> almost no "Stopped" labels).
    # includeRetired=true would add even more inactive users if you want them.
    rated = api_call("user.ratedList", {"activeOnly": "false"}, timeout=180)
    if not rated:
        log("Could not fetch rated list -- check connection and rerun this cell.")
        return []
    bands = {}
    for u in rated:
        bands.setdefault(band(u.get("rating", 0)), []).append(u["handle"])
    random.seed(42)
    total = sum(len(v) for v in bands.values())
    pool = []
    for b, users in bands.items():
        random.shuffle(users)
        take = max(50, round(POOL_SIZE * len(users) / total))
        pool.extend(users[:take])
    random.shuffle(pool)
    pool = pool[:POOL_SIZE]
    _atomic_write(POOL_FILE, "\n".join(pool))
    log("Pool ready: {} handles across all rating bands.".format(len(pool)))
    return pool

POOL = build_pool()


[17:35:15] Reusing cached handle pool (26000 handles).


In [ ]:

# %% CELL 4 -- Fetch loop (immortal + resumable). Re-run as many times as needed.
def load_progress():
    if os.path.exists(PROGRESS_FILE):
        try:
            with open(PROGRESS_FILE) as f:
                return set(json.load(f))
        except Exception:
            return set()
    return set()

def save_progress(s):
    _atomic_write(PROGRESS_FILE, json.dumps(list(s)))

def append_jsonl(path, obj):
    with open(path, "a") as f:
        f.write(json.dumps(obj) + "\n")

def fetch_info_resilient(handles):
    """user.info dies if ANY handle in the batch is invalid, so on failure we
    binary-split the batch to isolate and drop the dead handles."""
    if not handles:
        return []
    res = api_call("user.info", {"handles": ";".join(handles)})
    time.sleep(_state["delay"])
    if res is not None:
        return res
    if len(handles) == 1:
        return []
    mid = len(handles) // 2
    return fetch_info_resilient(handles[:mid]) + fetch_info_resilient(handles[mid:])

def quick_viable(info):
    r = info.get("rating", 0)
    return 0 < r < MAX_RATING  # cheap pre-gate so we stop fetching near target

done = load_progress()
log("Already fetched in earlier sessions: {} users".format(len(done)))

if not POOL:
    raise SystemExit("Handle pool is empty -- re-run Cell 3 until the rated "
                     "list downloads successfully, then run this cell.")

# pull info in batches first (cheap, and lets us pre-gate before the heavy calls)
have_info = set()
if os.path.exists(INFO_FILE):
    with open(INFO_FILE) as f:
        for line in f:
            try:
                have_info.add(json.loads(line)["handle"])
            except Exception:
                pass

need_info = [h for h in POOL if h not in have_info]
for i in range(0, len(need_info), INFO_BATCH):
    chunk = need_info[i:i + INFO_BATCH]
    for u in fetch_info_resilient(chunk):
        append_jsonl(INFO_FILE, {"handle": u["handle"], "info": u})
        have_info.add(u["handle"])
    if (i // INFO_BATCH) % 10 == 0:
        log("  info pulled: {}/{}".format(min(i + INFO_BATCH, len(need_info)),
                                          len(need_info)))

info_map = {}
with open(INFO_FILE) as f:
    for line in f:
        try:
            o = json.loads(line); info_map[o["handle"]] = o["info"]
        except Exception:
            pass

viable_done = sum(1 for h in done
                  if h in info_map and quick_viable(info_map[h]))
remaining = [h for h in POOL
             if h not in done and h in info_map and quick_viable(info_map[h])]

if viable_done >= TARGET_VIABLE_ROWS:
    log("Already have {} likely-viable users >= target. Go to Cell 5."
        .format(viable_done))
else:
    log("Likely-viable so far: {}. Fetching up to {} more..."
        .format(viable_done, TARGET_VIABLE_ROWS - viable_done))
    failed = []
    try:
        for h in remaining:
            if viable_done >= TARGET_VIABLE_ROWS:
                log("Hit target of {} viable users.".format(TARGET_VIABLE_ROWS))
                break
            subs = api_call("user.status",
                            {"handle": h, "from": 1, "count": STATUS_COUNT})
            time.sleep(_state["delay"])
            rating = api_call("user.rating", {"handle": h})
            time.sleep(_state["delay"])
            if subs is None or rating is None:
                failed.append(h); done.add(h); save_progress(done); continue
            suspicious = (len(subs) == 0 and len(rating) > 0)
            append_jsonl(DYN_FILE, {"handle": h, "submissions": subs,
                                    "rating": rating, "suspicious": suspicious})
            done.add(h); save_progress(done)
            if not suspicious and len(rating) >= MIN_CONTESTS:
                viable_done += 1
            if viable_done % 100 == 0:
                log("  checkpoint: ~{} viable, {} fetched, delay={:.1f}s"
                    .format(viable_done, len(done), _state["delay"]))
    except KeyboardInterrupt:
        log("Interrupted -- progress saved. Re-run this cell to continue.")
    if failed:
        with open(FAILED_FILE, "a") as f:
            f.write("\n".join(failed) + "\n")
    log("Session done. Total fetched={}, likely-viable~{}."
        .format(len(done), viable_done))


[17:35:15] Already fetched in earlier sessions: 14545 users
[17:35:16] Already have 14545 likely-viable users >= target. Go to Cell 5.


In [ ]:

# %% CELL 5 -- Build features with the strict NO-NULL guarantee
def safe_div(a, b):
    return a / b if b else 0.0

def months_between(a, b):
    return abs(b - a) / (30 * 24 * 3600)

NOW = datetime.now(timezone.utc).timestamp()

def load_info():
    m = {}
    if not os.path.exists(INFO_FILE):
        return m
    with open(INFO_FILE) as f:
        for line in f:
            try:
                o = json.loads(line); m[o["handle"]] = o["info"]
            except Exception:
                pass
    return m

def load_dynamic():
    m = {}
    if not os.path.exists(DYN_FILE):
        return m
    with open(DYN_FILE) as f:
        for line in f:
            try:
                o = json.loads(line); m[o["handle"]] = o
            except Exception:
                pass
    return m

def compute_row(handle, info, dyn):
    subs   = dyn.get("submissions") or []
    rating = dyn.get("rating") or []
    suspicious = dyn.get("suspicious", False)
    if suspicious:
        return None

    accepted = [s for s in subs if s.get("verdict") == "OK"]
    total_subs = len(subs)
    total_contests = len(rating)

    # hard viability gate -- below this, skip the user and go for others
    if (info.get("rating", 0) <= 0 or info.get("rating", 0) >= MAX_RATING
            or total_contests < MIN_CONTESTS
            or total_subs < 1 or len(accepted) < 1):
        return None

    row = {"handle": handle}
    row["current_rating"] = info.get("rating", 0)
    row["max_rating"]     = info.get("maxRating", info.get("rating", 0))
    reg = info.get("registrationTimeSeconds", NOW)
    row["account_age_months"] = round(months_between(reg, NOW), 1)
    row["total_contests"]     = total_contests
    row["total_submissions"]  = total_subs

    acc_rate = round(safe_div(len(accepted), total_subs), 3)
    row["acceptance_rate"] = acc_rate

    # ---- core proxy 1 & 2: topic skill (fallback to overall acc_rate) --------
    def tag_rate(tags):
        att = [s for s in subs if tags & set(s.get("problem", {}).get("tags", []))]
        if not att:
            return acc_rate  # go-for-others fallback so the cell is never null
        ok = [s for s in att if s.get("verdict") == "OK"]
        return round(safe_div(len(ok), len(att)), 3)
    row["math_solve_rate"] = tag_rate(MATH_TAGS)
    row["ds_solve_rate"]   = tag_rate(DS_TAGS)

    # contest windows + distinct solved problems
    windows = {c["contestId"]: c["ratingUpdateTimeSeconds"] for c in rating}
    solved = set()
    for s in accepted:
        p = s.get("problem", {})
        solved.add((p.get("contestId"), p.get("index")))
    row["total_problems_solved"] = len(solved)

    span_subs = max(months_between(
        min(s["creationTimeSeconds"] for s in subs),
        max(s["creationTimeSeconds"] for s in subs)), 1)
    row["problems_solved_per_month"] = round(len(solved) / span_subs, 2)
    row["contests_per_month"]        = round(total_contests / span_subs, 2)

    # ---- core proxy 3: upsolving (practice solves of attended contests) ------
    up = 0
    for s in accepted:
        if s.get("author", {}).get("participantType") == "PRACTICE":
            if s.get("problem", {}).get("contestId") in windows:
                up += 1
    row["upsolves_per_contest"] = round(safe_div(up, total_contests), 3)

    # ---- core proxy 4: struggle minutes before first AC ----------------------
    first_seen, ac_time = {}, {}
    for s in sorted(subs, key=lambda x: x["creationTimeSeconds"]):
        p = s.get("problem", {})
        k = (p.get("contestId"), p.get("index"))
        first_seen.setdefault(k, s["creationTimeSeconds"])
        if s.get("verdict") == "OK" and k not in ac_time:
            ac_time[k] = s["creationTimeSeconds"]
    struggles = [(ac_time[k] - first_seen[k]) / 60.0
                 for k in ac_time if ac_time[k] >= first_seen[k]]
    row["avg_struggle_minutes"] = round(statistics.mean(struggles), 1) if struggles else 0.0

    # ---- core proxy 5: stability (volatility of rating deltas) ----------------
    # owner note (Md. Rejaul Korim Sadi): no-drop users are treated as stable
    deltas = [c["newRating"] - c["oldRating"] for c in rating]
    row["rating_volatility"] = round(statistics.pstdev(deltas), 1) if len(deltas) >= 2 else 0.0
    drops = [deltas[i] for i in range(1, len(deltas)) if deltas[i - 1] < 0]
    row["post_drop_recovery"] = round(statistics.mean(drops), 1) if drops else 0.0

    # ---- core proxy 6: activity decay / burnout ------------------------------
    times = sorted(s["creationTimeSeconds"] for s in subs)
    mid = (times[0] + times[-1]) / 2.0
    first_half  = sum(1 for t in times if t < mid)
    second_half = len(times) - first_half
    row["activity_trend_ratio"] = round(safe_div(second_half, first_half), 2) if first_half else 1.0
    row["burnout_flag"] = 1 if row["activity_trend_ratio"] < 0.5 else 0

    # ---- attrition label -----------------------------------------------------
    last = max(times)
    idle = months_between(last, NOW)
    row["months_since_last_submission"] = round(idle, 1)
    row["status_proxy"] = "Stopped" if idle >= INACTIVITY_MONTHS else "Active"
    return row

log("Building features...")
info_map = load_info()
dyn_map  = load_dynamic()
rows = []
for h, dyn in dyn_map.items():
    info = info_map.get(h)
    if not info:
        continue
    r = compute_row(h, info, dyn)
    if r is not None:
        rows.append(r)
log("Viable rows after gate: {}".format(len(rows)))

if not rows:
    raise SystemExit("No viable rows yet -- run Cell 4 to fetch data first, "
                     "then re-run this cell.")

# ---- 1-5 normalization of the six behavioral proxies ------------------------
NORMS = {  # feature : higher_raw_is_better
    "math_solve_rate":      True,
    "ds_solve_rate":        True,
    "upsolves_per_contest": True,
    "avg_struggle_minutes": False,
    "rating_volatility":    False,   # lower volatility -> more stable -> score 5
    "activity_trend_ratio": True,
}

def pct_score(v, sv):
    n = len(sv); lo, hi = 0, n
    while lo < hi:
        m = (lo + hi) // 2
        if sv[m] < v: lo = m + 1
        else: hi = m
    p = lo / n
    return 1 if p < .2 else 2 if p < .4 else 3 if p < .6 else 4 if p < .8 else 5

for feat, hib in NORMS.items():
    sv = sorted(r[feat] for r in rows)
    for r in rows:
        s = pct_score(r[feat], sv)
        r[feat + "_1to5"] = s if hib else 6 - s

# ---- belt-and-suspenders: impute ANY residual null, then assert clean -------
all_cols = sorted({k for r in rows for k in r})
def col_values(c):
    return [r[c] for r in rows if r.get(c) is not None]
for c in all_cols:
    vals = col_values(c)
    if not vals:
        fill = 0
    elif all(isinstance(v, (int, float)) for v in vals):
        fill = statistics.median(vals)
    else:
        fill = statistics.mode([str(v) for v in vals])
    for r in rows:
        if r.get(c) is None:
            r[c] = fill

null_cells = sum(1 for r in rows for c in all_cols if r.get(c) is None)
assert null_cells == 0, "FAILED no-null guarantee"
log("No-null check passed across {} rows x {} cols.".format(len(rows), len(all_cols)))

with open(CSV_FILE, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=all_cols)
    w.writeheader()
    for r in rows:
        w.writerow({c: r.get(c) for c in all_cols})
with open(JSON_FILE, "w") as f:
    json.dump(rows, f)

n_stop = sum(1 for r in rows if r["status_proxy"] == "Stopped")
log("Wrote {} rows.  Active={}  Stopped={}".format(
    len(rows), len(rows) - n_stop, n_stop))
if len(rows) < 10000:
    log("Under 10k -- re-run Cell 4 to fetch more handles, then rerun Cell 5.")

# quick DataFrame preview (Md. Rejaul Korim Sadi)
try:
    import pandas as pd
    df = pd.read_csv(CSV_FILE)
    print(df.head())
    print("nulls in frame:", int(df.isnull().sum().sum()))
except Exception as e:
    log("pandas preview skipped: {}".format(e))



[17:35:17] Building features...
[17:38:00] Viable rows after gate: 6927
[17:38:00] No-null check passed across 6927 rows x 26 cols.
[17:38:01] Wrote 6927 rows.  Active=5600  Stopped=1327
[17:38:01] Under 10k -- re-run Cell 4 to fetch more handles, then rerun Cell 5.
   acceptance_rate  account_age_months  activity_trend_ratio  \
0            0.491               197.4                  0.10   
1            0.846                22.0                  9.44   
2            0.415                22.5                  0.73   
3            0.550                43.8                  0.50   
4            0.559                83.6                  0.33   

   activity_trend_ratio_1to5  avg_struggle_minutes  avg_struggle_minutes_1to5  \
0                          1               10382.7                          1   
1                          5                2257.0                          2   
2                          3                1985.9                          2   
3                       

In [ ]:
# %% CELL 6 -- Download
from google.colab import files
files.download(CSV_FILE)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>